In [11]:
# Configuration Parameters

#data folder is "data"

INPUT_FILE    = "data/raw/proteins.xlsx" 
CLEANSED_FILE = "data/processed/cleansed..xlsx"
OUPTPUT_FILE  = "data/final/pair_wise_correlations.xlsx"

SPECIFIC_PROTEINS = ['Actb', 'Bp180', 'Itgb4']


In [12]:
# Two folders: notebooks and data
#     notebooks:
#        this Jupyter NOtebook
#     data:
#       /raw has the input file
#       /cleansed has the cleansed input file
#       /final has the final output as various worksheets
#     see above cell for configuration parameters

# Process pipeline: 
#       Read the input file, cleanse, do pairwise correlation, write to output
#       Additionally, write to the output pairwise correlation for specific proteins - one/worksheet.
#       Configuration cell parameter, SPECIFIC_PROTEINS, notes the specific proteins

#1 replace none with np.nan & save as excel file
#2 do pair-wise correlation [and print corr matrix]
#3 save all 3 worksheets (input, cleansed, and corr matrix) to an outputput excel file
#4 convert corr matrix to long format
#5 (works fine)  against old data from 2024, "4 old data," worksheet "1 Orig DS"
#6 write rows matching specific proteinS (from a list) to separate Excel sheets

import pandas as pd
import numpy as np
from pathlib import Path
import os

parentDir = parent = Path.cwd().parent 

excelInputFile = parentDir / INPUT_FILE
excelCleansedFile = parentDir / CLEANSED_FILE
excelOutputFile = parentDir / OUPTPUT_FILE 

#1 read input file and copy to output as well
df = pd.read_excel(excelInputFile )
df.to_excel(excelOutputFile, sheet_name="1 Original DS", index=False)

#2 replace none with np.nan 
replWithNan = ["none"] # can add more if you like
df2 = df.replace(replWithNan, np.nan)
df2.to_excel(excelCleansedFile, sheet_name='2 Cleansed DS', index=False)

#3 do pair-wise correlation [and print corr matrix]
#        Ensure at least (N - 1) valid pairs!
#        Any pair with 2 or more nulls will fall below this threshold and return NaN
N = len(df2)
corrMatrixDF = df2.corr(method='pearson', min_periods=N - 1)
#print (df2, '\n', corrMatrixDF )

# Set self-correlations (diagonal) to NaN
np.fill_diagonal(corrMatrixDF.values, np.nan)

#4 convert corr matrix to long format
longFormatDF = corrMatrixDF.stack().reset_index()

#Rename columns for clarity
longFormatDF.columns = ['Protein 1', 'Protein 2', 'Pearson']
#print (longFormatDF)

#5 Append cleansed input and corr matrix to output. AS well, write all other worksheets!
with pd.ExcelWriter(excelOutputFile, mode='a', engine='openpyxl') as writer:
    df2.to_excel(writer, sheet_name='2 Cleansed DS', index=False)  #cleansed sheet
    corrMatrixDF.to_excel(writer, sheet_name='3 Correlation')  #cleansed sheet , index=True
    longFormatDF.to_excel(writer, sheet_name='4 Correlation long format', index=False)  #cleansed sheet , index=True

    #6 for each protein in SPECIFIC_PROTEINS list, write its correlation as a separate Excel sheet
    targetProteins = SPECIFIC_PROTEINS
    df = longFormatDF
    
    worksheetNo = 5
    for item in targetProteins:
        # Filter DataFrame where the Category column matches the item
        matchedProteins = df[df['Protein 1'] == item]
        matchedProteins = matchedProteins.sort_values(by='Pearson', ascending=False)
        
        # Write the filtered data to a sheet named after the item
        sheetName = str(worksheetNo) + " " + item + " Ordered"
        worksheetNo = worksheetNo + 1
        matchedProteins.to_excel(writer, sheet_name= sheetName, index=False)


C:\Users\kresm\AppData\Local\Temp\ipykernel_22300\3432847933.py:35: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  df.to_excel(excelOutputFile, sheet_name="1 Original DS", index=False)
C:\Users\kresm\AppData\Local\Temp\ipykernel_22300\3432847933.py:40: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  df2.to_excel(excelCleansedFile, sheet_name='2 Cleansed DS', index=False)
